In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
import lightgbm as lgb
from transformers import AutoModel, AutoTokenizer
import os

In [2]:
desc_tensor = torch.load("torch_data/descriptors.pt")

In [3]:
print(len(desc_tensor))
print(desc_tensor)

16648
tensor([[ 0.2666,  0.2666, -1.0064,  ..., -0.2238, -0.1981, -0.0439],
        [ 0.4235,  0.4235, -0.3401,  ..., -0.2238, -0.1981, -0.0439],
        [ 0.4412,  0.4412, -0.9842,  ..., -0.2238, -0.1981, -0.0439],
        ...,
        [ 0.7704,  0.7704, -0.1347,  ..., -0.2238, -0.1981, -0.0439],
        [ 0.7704,  0.7704, -0.1347,  ..., -0.2238, -0.1981, -0.0439],
        [ 0.7704,  0.7704, -0.1347,  ..., -0.2238, -0.1981, -0.0439]])


In [4]:
msl_new = pd.read_csv('input_data/msl_new.csv')
print(len(msl_new))
msl_new.head()

16648


,Chromophore,Solvent,Quantum yield
0,O=C([O-])c1ccccc1-c1c2ccc(=O)cc-2oc2cc([O-])ccc12,O,0.950
1,O=C([O-])c1ccccc1C1=c2cc3c4c(c2Oc2c1cc1c5c2CCC...,CO,1.000
2,O=C([O-])c1ccccc1-c1c2cc(Br)c(=O)c(Br)c-2oc2c(...,O,0.200
3,O=C([O-])c1ccccc1-c1c2cc(I)c(=O)c(I)c-2oc2c(I)...,O,0.020
4,O=C([O-])c1c(Cl)c(Cl)c(Cl)c(Cl)c1-c1c2cc(I)c(=...,O,0.018


In [5]:
model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct", deterministic_eval=True, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct", trust_remote_code=True)


In [6]:
import os
import torch

CACHE_CHROMOPHORES = "embedding_caches/chromophores.pt"
CACHE_SOLVENTS = "embedding_caches/solvents.pt"

def load_cache(cache_file):
    if os.path.exists(cache_file):
        return torch.load(cache_file)
    else:
        return {}  # empty cache

def save_cache(cache, cache_file):
    torch.save(cache, cache_file)

In [7]:
def get_embeddings(smiles_list, cache_file, batch_size=32, max_length=202):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    cache = load_cache(cache_file)
    all_embeddings = []


    to_compute = [s for s in smiles_list if s not in cache]

    if len(to_compute) > 0:
        print(f"Computing {len(to_compute)} new embeddings on {device}...")


        model.to(device)
        model.eval()

        with torch.no_grad():
            for i in range(0, len(to_compute), batch_size):
                batch = to_compute[i:i+batch_size]

                inputs = tokenizer(
                    batch, padding=True, truncation=True,
                    max_length=max_length, return_tensors="pt"
                ).to(device)


                outputs = model(**inputs)
                embeddings = outputs.pooler_output 


                for s, emb in zip(batch, embeddings):
                    cache[s] = emb.cpu()


        save_cache(cache, cache_file)

    else:
        print("All embeddings already in cache.")


    ordered = [cache[s] for s in smiles_list]
    return torch.stack(ordered)


In [8]:
vecs_mols = get_embeddings(msl_new["Chromophore"].tolist(), CACHE_CHROMOPHORES)
print("Chromophore embeddings shape:", vecs_mols.shape)

vecs_sols = get_embeddings(msl_new["Solvent"].tolist(), CACHE_SOLVENTS)
print("Solvent embeddings shape:", vecs_sols.shape)


All embeddings already in cache.
Chromophore embeddings shape: torch.Size([16648, 768])
All embeddings already in cache.
Solvent embeddings shape: torch.Size([16648, 768])


In [9]:
X = torch.cat([vecs_mols, vecs_sols, desc_tensor], dim=1)
mean = torch.nanmean(X)
X = X.nan_to_num(mean)
print(X.shape)

y = torch.tensor(msl_new['Quantum yield'].values)
print(y.shape)

torch.Size([16648, 1753])
torch.Size([16648])


In [10]:
import numpy as np
from sklearn.model_selection import train_test_split

X_np = X.cpu().numpy()
y_np = y.cpu().numpy()

X_train, X_temp, y_train, y_temp = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(13318, 1753) (13318,)
(1665, 1753) (1665,)
(1665, 1753) (1665,)


In [11]:
print(X_np.shape)
print(y_np.shape)

(16648, 1753)
(16648,)


In [12]:
np.save('model_data/X_desc.npy', X_np)
np.save('model_data/y_desc.npy', y_np)

In [13]:
mlp = MLPRegressor(
    hidden_layer_sizes=(512, 256, 128, 64),
    activation='relu',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=2000,
    random_state=42,
    warm_start=False,
    early_stopping=True
)

X_train_mlp = np.vstack([X_train, X_val])
y_train_mlp = np.concatenate([y_train, y_val])

mlp.fit(X_train_mlp, y_train_mlp)

y_test_pred = mlp.predict(X_test)
print("test:")
print(f"R2: {r2_score(y_test, y_test_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.3f}")

test:
R2: 0.625
RMSE: 0.185


In [14]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist"
)

eval_set = [(X_train, y_train), (X_val, y_val)]

xgb_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)

y_val_xgb = xgb_model.predict(X_val)
print("validation:")
print(f"R Squared: {r2_score(y_val, y_val_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_val_xgb)):.3f}")
print()
y_test_xgb = xgb_model.predict(X_test)
print("test:")
print(f"R Squared: {r2_score(y_test, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_xgb)):.3f}")

validation:
R Squared: 0.637
RMSE: 0.188

test:
R Squared: 0.684
RMSE: 0.170


In [15]:
lgb_model = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.1, random_state=42)

eval_set = [(X_train, y_train), (X_val, y_val)]

lgb_model.fit(X_train, y_train, eval_set=eval_set, eval_metric="rmse", callbacks=[lgb.early_stopping(stopping_rounds=50)])

y_val_lgb = lgb_model.predict(X_val)
print("validation:")
print(f"R Squared: {r2_score(y_val, y_val_lgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_val_lgb)):.3f}")
print()
print("test:")
y_test_lgb = lgb_model.predict(X_test)
print(f"R Squared: {r2_score(y_test, y_test_lgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_lgb)):.3f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.068372 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 377825
[LightGBM] [Info] Number of data points in the train set: 13318, number of used features: 1730
[LightGBM] [Info] Start training from score 0.344984
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1058]	training's rmse: 0.0262282	training's l2: 0.000687919	valid_1's rmse: 0.188397	valid_1's l2: 0.0354933
validation:
R Squared: 0.634
RMSE: 0.188

test:
R Squared: 0.686
RMSE: 0.169


/Users/utoglu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/utoglu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


### Fingerprints

In [16]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdFingerprintGenerator

In [17]:
fgp_tensor = torch.load('torch_data/fingerprints.pt')
print(len(fgp_tensor))
print(fgp_tensor)

16648
tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]], dtype=torch.int32)


In [18]:
X = torch.cat([vecs_mols, vecs_sols, desc_tensor, fgp_tensor], dim=1)
mean = torch.nanmean(X)
X = X.nan_to_num(mean)
print(X.shape)

y = torch.tensor(msl_new['Quantum yield'].values)
print(y.shape)

torch.Size([16648, 3801])
torch.Size([16648])


In [19]:
import numpy as np
from sklearn.model_selection import train_test_split

X_np = X.cpu().numpy()
y_np = y.cpu().numpy()

X_train, X_temp, y_train, y_temp = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(13318, 3801) (13318,)
(1665, 3801) (1665,)
(1665, 3801) (1665,)


In [20]:
print(X_np.shape)
print(y_np.shape)

(16648, 3801)
(16648,)


In [21]:
np.save('model_data/X_fgp.npy', X_np)
np.save('model_data/y_gfp.npy', y_np)

In [22]:
mlp = MLPRegressor(
    hidden_layer_sizes=(512, 256, 128, 64),
    activation='relu',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=2000,
    random_state=42,
    warm_start=False,
    early_stopping=True
)

X_train_mlp = np.vstack([X_train, X_val])
y_train_mlp = np.concatenate([y_train, y_val])

mlp.fit(X_train_mlp, y_train_mlp)

y_test_pred = mlp.predict(X_test)
print("test:")
print(f"R2: {r2_score(y_test, y_test_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.3f}")

test:
R2: 0.694
RMSE: 0.167


In [23]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_val, y_val)]

xgb_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)

y_val_xgb = xgb_model.predict(X_val)
print("validation:")
print(f"R Squared: {r2_score(y_val, y_val_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_val_xgb)):.3f}")
print()
y_test_xgb = xgb_model.predict(X_test)
print("test:")
print(f"R Squared: {r2_score(y_test, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_xgb)):.3f}")

/Users/utoglu/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [16:11:28] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)


validation:
R Squared: 0.654
RMSE: 0.183

test:
R Squared: 0.699
RMSE: 0.165


In [24]:
lgb_model = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.1, random_state=42)

eval_set = [(X_val, y_val)]

lgb_model.fit(X_train, y_train, eval_set=eval_set, eval_metric="rmse", callbacks=[lgb.early_stopping(stopping_rounds=50)])

y_val_lgb = lgb_model.predict(X_val)
print("validation:")
print(f"R Squared: {r2_score(y_val, y_val_lgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_val_lgb)):.3f}")
print()
print("test:")
y_test_lgb = lgb_model.predict(X_test)
print(f"R Squared: {r2_score(y_test, y_test_lgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_lgb)):.3f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.192820 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 385550
[LightGBM] [Info] Number of data points in the train set: 13318, number of used features: 3598
[LightGBM] [Info] Start training from score 0.344984
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1641]	valid_0's rmse: 0.182047	valid_0's l2: 0.0331411
validation:
R Squared: 0.658
RMSE: 0.182

test:
R Squared: 0.707
RMSE: 0.163


/Users/utoglu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/utoglu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
